<a href="https://colab.research.google.com/github/avionerman/neural-networks-msc/blob/main/S1_Flowers102_CLIP_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S1 — Fine-Grained Flower Retrieval with CLIP + Trainable Projection Head

**Goal:** Build a text → image retrieval system on Oxford Flowers-102 (102 flower species).
Given a query like *"a photo of a purple iris"*, retrieve the top-k matching flower images.

**Architecture:**
- **Retriever (frozen):** CLIP ViT-B/32 — produces image & text embeddings
- **Trainable component:** Small MLP projection head on top of CLIP image embeddings
- **Vector store:** FAISS (flat, cosine similarity)
- **Training:** Contrastive fine-tuning on class prompts (~30–60 minutes on T4)

**Pipeline:**
```
query text ──► CLIP text encoder ──► (normalize) ──► FAISS search
                                                         │
images ──► CLIP image encoder ──► Projection head ──► FAISS index
              (frozen)              (trained)
```

**What we evaluate:** Recall@1, Recall@5, Recall@10, and per-class mAP, before vs. after training.

> **Runtime setup (Colab):** Runtime → Change runtime type → **T4 GPU** (free) is enough.
> A100 / L4 / 3090 / 4090 will be 3–5× faster.

---
## Phase 1 — Environment setup

Install the small set of libraries we need. This takes ~1 minute.

In [ ]:
!pip install -q transformers datasets faiss-cpu accelerate peft torch torchvision Pillow tqdm matplotlib scikit-learn

In [ ]:
import os, random, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset
import faiss
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED);
np.random.seed(SEED);
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

---
## Phase 2 — Load Oxford Flowers-102

We use the HuggingFace `nelorth/oxford-flowers` mirror. It contains ~8,000 images across 102 classes.
If that mirror is unreachable, fall back to torchvision's `Flowers102` which downloads from the Oxford website.

Each sample has `image` (PIL) and `label` (int 0–101).

In [ ]:
# using the HuggingFace portal
ds = load_dataset("nelorth/oxford-flowers")
print("loaded from HuggingFace mirror")
print(ds)

In [ ]:
# source 1:
# https://www.robots.ox.ac.uk/~vgg/data/flowers/102/
# and
# source 2:
# https://github.com/jimgoo/caffe-oxford102/blob/master/class_labels.py

FLOWER_CLASSES = [
    "pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
    "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
    "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
    "yellow iris", "globe-flower", "purple coneflower", "peruvian lily", "balloon flower",
    "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger",
    "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian",
    "artichoke", "sweet william", "carnation", "garden phlox", "love in the mist",
    "mexican aster", "alpine sea holly", "ruby-lipped cattleya", "cape flower",
    "great masterwort", "siam tulip", "lenten rose", "barbeton daisy", "daffodil",
    "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold", "buttercup",
    "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula", "sunflower",
    "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
    "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan",
    "silverbush", "californian poppy", "osteospermum", "spring crocus", "bearded iris",
    "windflower", "tree poppy", "gazania", "azalea", "water lily", "rose", "thorn apple",
    "morning glory", "passion flower", "lotus", "toad lily", "anthurium", "frangipani",
    "clematis", "hibiscus", "columbine", "desert-rose", "tree mallow", "magnolia",
    "cyclamen", "watercress", "canna lily", "hippeastrum", "bee balm", "ball moss",
    "foxglove", "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia",
    "blanket flower", "trumpet creeper", "blackberry lily",
]

assert len(FLOWER_CLASSES) == 102
print(f"loaded {len(FLOWER_CLASSES)} class names")
print("some examples ->", FLOWER_CLASSES[:5])

In [ ]:
# just testing that the training set images are getting printed
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
split_key = "train"
print(split_key + ": ")
for i, ax in enumerate(axes):
    sample = ds[split_key][i]
    ax.imshow(sample["image"])
    ax.set_title(FLOWER_CLASSES[sample["label"]], fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

# just testing that the testing set images are getting printed
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
split_key = "test"
print("\n" + split_key + ": ")
for i, ax in enumerate(axes):
    sample = ds[split_key][i]
    ax.imshow(sample["image"])
    ax.set_title(FLOWER_CLASSES[sample["label"]], fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

---
## Phase 3 — Load CLIP (frozen)

CLIP ViT-B/32 from OpenAI. We freeze it entirely and only use its forward pass to extract embeddings.
The model has ~151M parameters but we never backpropagate through it, so memory is modest.

In [ ]:
# [A] organization -> openai

# [B] model family -> clip (contrastive language-image pre-training)
# the clip trains an image encoder and a text encoder together so to
# match image-text pairs so to end up close in the vector space

# [C] encoder type -> vit (Vision Transf.). So clip can use diff image encoders
# the vit

# [D] model size -> base - so the base size will be used to keep a balance
# there are tiny, small, large and huge sizes too

# [E] patch versions -> 32 will cut the image in 32x32 pixel chunkcs
# before any Transformer goes to the photos, CLIP resizes it to a fixed size
# which is 224 x 224. ViT will try to turn this into tokens using patches
# so each patch will be 32x32 pixels
# i.e., 224 / 32 = 7 for the width and another 7 for the height
# that means we will have 7 * 7 = 49 patches in total (x1 patch -> x1 token)

CLIP_CKPT = "openai/clip-vit-base-patch32"

device = "cuda" if torch.cuda.is_available() else "cpu"

# downloads the clip weights from huggingface and loads them
# the clip_model now contains both a Vision and a Text Transformer
clip_model = CLIPModel.from_pretrained(CLIP_CKPT).to(device)

# loads model's pre processor
# conversts raw inputs to what the model actually needs
# - For images will be resized to 224x224 and pixels will be normalized
# - For text will be tokenized into numbers that the model understands
clip_processor = CLIPProcessor.from_pretrained(CLIP_CKPT)

# evaluation mode is ON, training is OFF (no dropout)
clip_model.eval()


for p in clip_model.parameters():
    # weights are not allowed to change during training due to 'False'
    p.requires_grad_(False)

# the length of the model is stored under it (output vector size)
# hardcoded by openai, and it's 512
EMB_DIM = clip_model.config.projection_dim
print(f"embedding dim -> {EMB_DIM}")

---
## Phase 4 — Compute image embeddings (one-time, cached)

Run every image through CLIP once and cache the embeddings. This is the "index build" step and
the most expensive part of the pipeline — ~2–5 minutes on T4 for all 8k images.

We'll save embeddings to disk so retraining / reloading doesn't redo this.

In [ ]:
# defining how I will encode

# we will follow batches since we can't load 7169 images in the GPU at once

# asking from PyTorch to not track gradients inside the function
# disabling it, cuts memory and increase the speed
# I'm not training so gradient is not needed here
@torch.no_grad()
def encode_images(pil_images, batch_size=64): # processing 64 imgs per batch
    all_emb = []
    for i in tqdm(range(0, len(pil_images), batch_size), desc="Encoding images"):

        # getting the next 64 imgs from the list
        # converts each to RGB and store the new one under 'batch'
        batch = [img.convert("RGB") for img in pil_images[i:i+batch_size]]

        # giving the preprocessor 64 imgs to process through 'batch'
        # requesting to to send them back as pt - PyTorch
        inputs = clip_processor(images=batch, return_tensors="pt").to(device)

        # pick the vision model from clip (ignoring the text atm)
        # the vision model attr req -> 'pixel_values'
        # inputs["pixel_values"] = (64, 3, 224, 224) from the previous step
        vision_out = clip_model.vision_model(pixel_values=inputs["pixel_values"])

        # pooled = (64, 768) which means that
        # we have 64 imgs represented by one 768-number vector
        # where 768 is a design choice backed into model's config
        pooled = vision_out.pooler_output

        # visual_projection() is a linear layer that make the numbers from
        # 768 -> 512
        # that's because vit is producing a 768 dim vector, but clip shared
        # a space of 512 dim
        # emb = (64, 512)
        emb    = clip_model.visual_projection(pooled)

        # shrinks each vector so the length becomes 1
        # for example if a vector = [3, 4, 0]
        # using pythagorean, this would be sqrt(25) = 5
        # so from a v of [x, y, z] -> w, where w is the LENGTH
        # to make it 1, we have to do: [3, 4, 0] / 5 = [0.6, 0.8, 0.0]
        # => sqrt(1) = 1
        # so the NEW vector will be [0.6, 0.8, 0.0] pontinig at the same dir
        # as the previous vector, though having size 1.
        # *also, dim=-1 in the () means work per row (axis)
        emb    = F.normalize(emb, dim=-1)

        #
        all_emb.append(emb.cpu())

        # putting them batches all together (7169 rows, 512-num vector per img)
    return torch.cat(all_emb, dim=0)

@torch.no_grad()

# text cheaper than the image, so we can increase from 64 to 128
def encode_texts(texts, batch_size=128):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # need to force UNIFORM length for the given sentence
        # padding: making short sentences longer
        # for example, if we have sent/s with 1, 3 ad 6, we pick 6 as max
        # and we add 0's to 'fill' the rest of the sent/s

        # trancation: make long sentences shorter
        # if query is more than 77 tokens, chops off everything after 77
        inputs = clip_processor(text=batch, return_tensors="pt", padding=True, truncation=True).to(device)

        # **inputs equals to:
        # input_ids=inputs["input_ids"],
        # attention_mask=inputs["attention_mask"]
        text_out = clip_model.text_model(**inputs)
        pooled = text_out.pooler_output
        emb    = clip_model.text_projection(pooled)
        emb    = F.normalize(emb, dim=-1)
        all_emb.append(emb.cpu())
    return torch.cat(all_emb, dim=0)

In [ ]:
# loading the raw data

# memorize the train data
train_images = [ds["train"][i]["image"] for i in range(len(ds["train"]))]
train_labels = np.array([ds["train"][i]["label"] for i in range(len(ds["train"]))])

# test split for evaluation
test_images = [ds["test"][i]["image"] for i in range(len(ds["test"]))]
test_labels = np.array([ds["test"][i]["label"] for i in range(len(ds["test"]))])

print(f"train: {len(train_images)}  test: {len(test_images)}")

In [ ]:
t0 = time.time()
train_emb = encode_images(train_images)   # (N_train, 512) float32 on CPU
test_emb  = encode_images(test_images)    # (N_test, 512)
print(f"encoding took {time.time()-t0:.1f}s")
print("train_emb:", train_emb.shape, "test_emb:", test_emb.shape)

# saving a PyTorch file for reuse
torch.save({"train_emb": train_emb, "train_labels": train_labels,
            "test_emb":  test_emb,  "test_labels":  test_labels},
           "clip_flowers_emb.pt")

---
## Phase 5 — Zero-shot baseline with FAISS (no training yet)

Before we train anything, evaluate CLIP out-of-the-box. This is crucial: if the baseline is already
near-perfect, we need a harder task; if it's weak, we have room to show improvement.

**Evaluation protocol — classification via retrieval.** For each test image, take its CLIP embedding
as the query, retrieve the top-k from the *train* set, and predict the majority class. Report top-1
accuracy (= Recall@1 in the "any correct-class neighbor" sense).

In [ ]:
def build_faiss(embeddings):
    emb = embeddings.numpy().astype(np.float32)
    idx = faiss.IndexFlatIP(emb.shape[1])   # inner product == cosine (already normalized)
    idx.add(emb)
    return idx

def retrieve(index, query_emb, k=5):
    q = query_emb.numpy().astype(np.float32)
    scores, ids = index.search(q, k)
    return scores, ids

def top1_accuracy(index, query_emb, query_labels, gallery_labels, k=1):
    _, ids = retrieve(index, query_emb, k=k)
    pred = gallery_labels[ids[:, 0]]
    return (pred == query_labels).mean()

def recall_at_k(index, query_emb, query_labels, gallery_labels, k=5):
    _, ids = retrieve(index, query_emb, k=k)
    hits = (gallery_labels[ids] == query_labels[:, None]).any(axis=1)
    return hits.mean()

In [ ]:
baseline_index = build_faiss(train_emb)
print("Gallery size:", baseline_index.ntotal)

baseline_metrics = {
    "Recall@1":  top1_accuracy(baseline_index, test_emb, test_labels, train_labels, k=1),
    "Recall@5":  recall_at_k(baseline_index, test_emb, test_labels, train_labels, k=5),
    "Recall@10": recall_at_k(baseline_index, test_emb, test_labels, train_labels, k=10),
}
print("Zero-shot baseline (frozen CLIP):")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Text → image retrieval baseline with class-name prompts
prompts = [f"a photo of a {c}" for c in FLOWER_CLASSES]
text_emb = encode_texts(prompts)
print("Text prompt embeddings:", text_emb.shape)

# For each test image, retrieve the nearest text prompt. The predicted class = argmax similarity.
sims = test_emb @ text_emb.T   # (N_test, 102)
pred = sims.argmax(dim=1).numpy()
zeroshot_acc = (pred == test_labels).mean()
print(f"CLIP zero-shot classification accuracy (text prompts): {zeroshot_acc:.4f}")

---
## Phase 6 — Define a trainable projection head

A tiny 2-layer MLP that sits on top of frozen CLIP image embeddings. We train it contrastively
so that images of the same flower species get mapped closer together and different species farther apart.

This is the ONLY component with gradients. Total trainable params ≈ 525k — trains in minutes.

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, dim=512, hidden=512, out=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out),
        )
        # Residual: start close to identity so we don't destroy CLIP's features
        self.alpha = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        y = self.net(x)
        out = x + self.alpha * y
        return F.normalize(out, dim=-1)

head = ProjectionHead(EMB_DIM, 512, EMB_DIM).to(device)
n_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

---
## Phase 7 — Train with supervised contrastive loss

We use a **supervised contrastive** objective: within a batch, embeddings of the same class should
be pulled together, embeddings of different classes pushed apart. Because we already have
pre-computed CLIP embeddings, each epoch is essentially a dense matmul — trivially fast.

Expected training time: **~5–10 minutes on T4**, ~2 minutes on A100.

In [ ]:
class EmbDataset(Dataset):
    def __init__(self, emb, labels):
        self.emb = emb
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.emb)
    def __getitem__(self, i): return self.emb[i], self.labels[i]

def sup_con_loss(z, labels, temperature=0.1):
    # z: (B, D) unit-norm; labels: (B,)
    sim = z @ z.T / temperature                 # (B, B)
    B = z.size(0)
    mask_self = torch.eye(B, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(mask_self, -1e9)
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos = (labels[:, None] == labels[None, :]) & ~mask_self  # (B, B)
    pos_count = pos.sum(dim=1).clamp(min=1)
    loss = -(log_prob * pos.float()).sum(dim=1) / pos_count
    return loss.mean()

In [ ]:
# Hyperparameters — tuned for fast convergence
BATCH = 256
EPOCHS = 20
LR = 1e-3
WD = 1e-4

train_ds = EmbDataset(train_emb, train_labels)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True)

opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(train_loader))

history = []
t0 = time.time()
for epoch in range(EPOCHS):
    head.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(device); yb = yb.to(device)
        zb = head(xb)
        loss = sup_con_loss(zb, yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
    mean_loss = float(np.mean(losses))

    # Quick eval every epoch
    head.eval()
    with torch.no_grad():
        tr = head(train_emb.to(device)).cpu()
        te = head(test_emb.to(device)).cpu()
    idx = build_faiss(tr)
    r1 = top1_accuracy(idx, te, test_labels, train_labels, k=1)
    history.append({"epoch": epoch, "loss": mean_loss, "R@1": r1})
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={mean_loss:.4f}  R@1={r1:.4f}")
print(f"Total training time: {time.time()-t0:.1f}s")

In [ ]:
# Plot training curves
fig, ax1 = plt.subplots(figsize=(8,4))
ep = [h["epoch"] for h in history]
ax1.plot(ep, [h["loss"] for h in history], "b-", label="loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss", color="b")
ax2 = ax1.twinx()
ax2.plot(ep, [h["R@1"] for h in history], "r-", label="R@1")
ax2.set_ylabel("Recall@1", color="r")
plt.title("Training curves"); plt.tight_layout(); plt.show()

---
## Phase 8 — Re-evaluate after training

Compare trained-projection metrics against the frozen baseline. Expect meaningful gains in Recall@1
on the class-retrieval task because the head has learned to cluster same-species images.

In [ ]:
head.eval()
with torch.no_grad():
    tr_proj = head(train_emb.to(device)).cpu()
    te_proj = head(test_emb.to(device)).cpu()

trained_index = build_faiss(tr_proj)
trained_metrics = {
    "Recall@1":  top1_accuracy(trained_index, te_proj, test_labels, train_labels, k=1),
    "Recall@5":  recall_at_k(trained_index, te_proj, test_labels, train_labels, k=5),
    "Recall@10": recall_at_k(trained_index, te_proj, test_labels, train_labels, k=10),
}

print(f"{'Metric':<12}{'Baseline':>12}{'Trained':>12}{'Δ':>10}")
for k in baseline_metrics:
    b, t = baseline_metrics[k], trained_metrics[k]
    print(f"{k:<12}{b:>12.4f}{t:>12.4f}{t-b:>+10.4f}")

---
## Phase 9 — Demo: text query → retrieve flower images

End-to-end interactive retrieval. Try different prompts and see what comes back.

In [ ]:
@torch.no_grad()
def text_to_images(query, k=5, use_trained_head=True):
    # robust text encoding (no get_text_features)
    t_inp = clip_processor(text=[query], return_tensors="pt", padding=True).to(device)
    text_out = clip_model.text_model(**t_inp)
    q = clip_model.text_projection(text_out.pooler_output)
    q = F.normalize(q, dim=-1)
    if use_trained_head:
        q = head(q)
        gallery = tr_proj
    else:
        gallery = train_emb
    idx = build_faiss(gallery)
    scores, ids = idx.search(q.cpu().numpy().astype(np.float32), k)
    return scores[0], ids[0]

print("Patched text_to_images.")

In [ ]:
# Try your own queries
show_retrieval("a pink rose in bloom", k=5)
show_retrieval("a white flower with long petals", k=5)
show_retrieval("a red flower with green leaves", k=5)

---
## Phase 10 — Ablation: Recall@k sweep

Show how performance varies with k. Good for your report.

In [ ]:
ks = [1, 3, 5, 10, 20, 50]
rows = []
for k in ks:
    r_base = recall_at_k(baseline_index, test_emb, test_labels, train_labels, k=k)
    r_tr   = recall_at_k(trained_index,  te_proj,  test_labels, train_labels, k=k)
    rows.append((k, r_base, r_tr))
    print(f"k={k:3d}  baseline={r_base:.4f}  trained={r_tr:.4f}")

plt.figure(figsize=(7,4))
plt.plot([r[0] for r in rows], [r[1] for r in rows], "o-", label="frozen CLIP")
plt.plot([r[0] for r in rows], [r[2] for r in rows], "s-", label="CLIP + projection")
plt.xlabel("k"); plt.ylabel("Recall@k"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Recall@k on Flowers-102 test set")
plt.show()

---
## Phase 11 — Save everything for your report

Dumps metrics, training history, and model weights to disk.

In [ ]:
import json
results = {
    "baseline":  {k: float(v) for k,v in baseline_metrics.items()},
    "trained":   {k: float(v) for k,v in trained_metrics.items()},
    "zeroshot_text_classification_acc": float(zeroshot_acc),
    "history": history,
    "recall_at_k_sweep": [{"k": k, "baseline": float(b), "trained": float(t)} for k,b,t in rows],
    "config": {
        "clip_ckpt": CLIP_CKPT, "epochs": EPOCHS, "batch": BATCH, "lr": LR,
        "embedding_dim": EMB_DIM, "n_train": len(train_images), "n_test": len(test_images),
    },
}
with open("s1_results.json", "w") as f:
    json.dump(results, f, indent=2)

torch.save(head.state_dict(), "s1_projection_head.pt")
print("Saved: s1_results.json, s1_projection_head.pt")
print(json.dumps(results["baseline"], indent=2))
print(json.dumps(results["trained"], indent=2))

---
## What to put in your report

1. **Problem & motivation** (½ page): fine-grained retrieval, why CLIP alone isn't enough, RAG framing.
2. **Method**:
   - Frozen CLIP as retriever (Transformer #1: ViT image encoder; Transformer #2: CLIP text encoder)
   - Trainable projection head (your learned component)
   - FAISS index for retrieval (the "R" in RAG)
3. **Training**: supervised contrastive on pre-computed CLIP features — cheap, fast.
4. **Results table** (Phase 8) — baseline vs. trained.
5. **Ablation** (Phase 10) — Recall@k curve.
6. **Qualitative examples** (Phase 9) — screenshots of retrieval results.
7. **Discussion**: what the head learned, where it fails, what you'd do with more time (e.g., LoRA on CLIP itself, add a BLIP caption generator to turn retrievals into descriptions → full RAG).

**Transformers covered in the pipeline:** ViT image encoder, CLIP text transformer — both transformer-based. That satisfies the course requirement clearly.